# MOLTIE – Prework Cell (Y Summary + JSONL Flatten)

## Purpose

This notebook cell performs **two prep steps** before deeper Moltie analysis:

1. **Summarise `Y_inferred.json`**: which WS rows produced a valid `Y` and which `X` tests were emitted.
2. **Flatten `MASTER_moltie.jsonl`** into tabular form and build a **PDF hit frequency** view.

It then saves both flattened outputs as CSVs next to the master JSONL file.

---

## Inputs

### Files

* `Y_inferred.json`

  * Path: `/home/hello/Projects/Statements/output/Y_inferred.json`
  * Expected to contain a `rows` object keyed by `y_row_id`.

* `MASTER_moltie.jsonl`

  * Path: `/home/hello/Projects/Statements/output/moltie_batch/MASTER_moltie.jsonl`

* `flatter.py`

  * Dir: `/home/hello/Projects/Statements/code/data_analysis`
  * Must export:

    * `flatten_moltie_jsonl(jsonl_path)`
    * `pdf_hit_frequency_view(df_flat)`

### Python deps

* `pandas`

---

## What it Produces

### 1) `y_summary_df`

A per-WS-row summary table:

* `y_row_id`: key from `Y_inferred.json: rows`
* `row_index_1based`: 1-based WS row index
* `y_ok`: whether inference succeeded for that row
* `n_x_tests`: number of `X*` tests emitted (only if `y_ok=True`)
* `x_keys`: list of X test keys (e.g. `['X5', 'X12']`)

Also prints:

* total Y rows
* count of `y_ok` vs `y_fail`

### 2) `df_flat`

Flattened table output from `MASTER_moltie.jsonl` (schema defined by `flatter.py`).

### 3) `df_pdf_freq`

Aggregated PDF-level frequency view computed from `df_flat`.

---

## Output Files

Saved next to `MASTER_moltie.jsonl`:

* `MASTER_moltie__flat.csv`
* `MASTER_moltie__pdf_frequency.csv`

(Implemented by stripping `.jsonl` then appending `__flat.csv` and `__pdf_frequency.csv`.)

---

## Execution Flow

1. **Validate input paths** with `assert`.
2. **Load Y JSON** and build `y_summary_df`:

   * Only extracts `x_tests` when `y_ok=True`.
3. **Import flatteners** by adding `FLATTER_DIR` to `sys.path`.
4. **Generate outputs**:

   * `df_flat = flatten_moltie_jsonl(JSONL_PATH)`
   * `df_pdf_freq = pdf_hit_frequency_view(df_flat)`
5. **Save CSVs** and display:

   * `df_flat.head(10)`
   * `df_pdf_freq.head(20)`

---

## Notes / Assumptions

* `Y_inferred.json` must have structure: `{'rows': {<y_row_id>: {...}}}`.
* `x_tests` is expected to be a dict; if not, it is ignored.
* `display(...)` assumes a Jupyter environment.


In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

# =========================================================
# PREWORK (Y + MOLTIE FLATTEN)
# 1) Summarise Y_inferred.json (which WS rows produced Y + which Xs)
# 2) Load MASTER_moltie.jsonl -> flatten to df_flat
# 3) Build pdf frequency view -> df_pdf_freq
# 4) Save both CSVs next to MASTER jsonl
# =========================================================

# -------------------------
# PATHS
# -------------------------
Y_PATH = Path("/home/hello/Projects/Statements/output/Y_inferred.json")
JSONL_PATH = Path("/home/hello/Projects/Statements/output/moltie_batch/MASTER_moltie.jsonl")
FLATTER_DIR = Path("/home/hello/Projects/Statements/code/data_analysis")  # contains flatter.py

assert Y_PATH.exists(), f"Missing: {Y_PATH}"
assert JSONL_PATH.exists(), f"Missing: {JSONL_PATH}"
assert (FLATTER_DIR / "flatter.py").exists(), f"Missing: {FLATTER_DIR/'flatter.py'}"

# -------------------------
# 1) Y SUMMARY (rows -> X-tests)
# -------------------------
with Y_PATH.open("r", encoding="utf-8") as f:
    y_data = json.load(f)

rows = y_data.get("rows", {})

records = []
for y_row_id, row_data in rows.items():
    y_ok = bool(row_data.get("y_ok", False))
    row_index = row_data.get("row_index_1based")

    x_keys = []
    if y_ok:
        y_block = row_data.get("y") or {}
        x_tests = y_block.get("x_tests") or {}
        if isinstance(x_tests, dict):
            x_keys = list(x_tests.keys())

    records.append({
        "y_row_id": y_row_id,
        "row_index_1based": row_index,
        "y_ok": y_ok,
        "n_x_tests": len(x_keys),
        "x_keys": x_keys,
    })

y_summary_df = (
    pd.DataFrame(records)
      .sort_values("row_index_1based", na_position="last")
      .reset_index(drop=True)
)

print("Y rows:", len(y_summary_df), "| y_ok:", int(y_summary_df["y_ok"].sum()), "| y_fail:", int((~y_summary_df["y_ok"]).sum()))
display(y_summary_df)

# -------------------------
# 2) Import flatter.py + build flat outputs
# -------------------------
if str(FLATTER_DIR) not in sys.path:
    sys.path.append(str(FLATTER_DIR))

from flatter import flatten_moltie_jsonl, pdf_hit_frequency_view  # noqa: E402

df_flat = flatten_moltie_jsonl(JSONL_PATH)
df_pdf_freq = pdf_hit_frequency_view(df_flat)

print("df_flat shape:", df_flat.shape)
print("df_pdf_freq shape:", df_pdf_freq.shape)

# -------------------------
# 3) Save next to MASTER jsonl
# -------------------------
BASE = JSONL_PATH.with_suffix("")  # remove .jsonl
CSV_FLAT = BASE.with_name(BASE.name + "__flat.csv")
CSV_PDF_FREQ = BASE.with_name(BASE.name + "__pdf_frequency.csv")

df_flat.to_csv(CSV_FLAT, index=False)
df_pdf_freq.to_csv(CSV_PDF_FREQ, index=False)

print("Saved flat:", CSV_FLAT)
print("Saved pdf frequency:", CSV_PDF_FREQ)


Y rows: 12 | y_ok: 10 | y_fail: 2


,y_row_id,row_index_1based,y_ok,n_x_tests,x_keys
0,X1_0001,1,False,0,[]
1,X1_0002,2,True,5,"[X1, X2, X3, X4, X5]"
2,X1_0003,3,True,7,"[X1, X2, X3, X4, X5, X6, X7]"
3,X1_0004,4,True,5,"[X1, X2, X3, X4, X5]"
4,X1_0005,5,True,5,"[X1, X2, X3, X4, X5]"
5,X1_0006,6,True,7,"[X1, X2, X3, X4, X5, X6, X7]"
6,X1_0007,7,True,5,"[X1, X2, X3, X4, X5]"
7,X1_0008,8,False,0,[]
8,X1_0009,9,True,4,"[X1, X2, X3, X4]"
9,X1_0010,10,True,7,"[X1, X2, X3, X4, X5, X6, X7]"


df_flat shape: (26853, 31)
df_pdf_freq shape: (2578, 12)
Saved flat: /home/hello/Projects/Statements/output/moltie_batch/MASTER_moltie__flat.csv
Saved pdf frequency: /home/hello/Projects/Statements/output/moltie_batch/MASTER_moltie__pdf_frequency.csv


# Needle Match Cell – WS vs Moltie Alignment

## Purpose

This cell classifies each witness-statement (WS) row into three structural buckets based on:

1. Whether it triggered any needle.
2. Whether it produced at least one relevant Moltie match.

It provides a high-level signal of retrieval effectiveness and argument resonance.

---

## Inputs

### 1) WS Enhanced CSV

`Leonardo_WS_enhanced.csv`

Used to determine:

* `y_row_id`
* Whether a row triggered any needle (`has__any_needle`, `needle_selected_raw`, or derived from `has__*` columns).

### 2) Moltie Flat Data

`df_flat` (output of `flatten_moltie_jsonl`)

Used to determine:

* Whether each `y_row_id` has at least one `relevant=True` result.

---

## Logic

### Step 1 – Determine Needle Presence

Each WS row is marked:

* `ws_has_any_needle = True` if any needle was triggered.
* Otherwise `False`.

### Step 2 – Determine Relevant Matches

From `df_flat`, compute per `y_row_id`:

* `has_relevant_case = True` if at least one Moltie result is `relevant=True`.

### Step 3 – Bucket Classification

Each WS row is assigned one of:

* `NEEDLE_AND_RELEVANT`
  Needle triggered and at least one relevant precedent found.

* `NEEDLE_BUT_NO_RELEVANT`
  Needle triggered but no relevant precedent found.

* `NO_NEEDLE`
  No needle triggered (row did not enter retrieval pipeline).

---

## Output

### 1) Frequency Table

For the three buckets:

* `count`
* `pct` (percentage of total WS rows)

This provides a structural health check of:

* Needle quality
* Retrieval precision
* Argument detectability

### 2) QA View

Optional preview of `ws_row` sorted by bucket for manual inspection.

---

## Interpretation

* High `NEEDLE_AND_RELEVANT` → Strong alignment between WS structure and precedent corpus.
* High `NEEDLE_BUT_NO_RELEVANT` → Needle too broad or X mapping misaligned.
* High `NO_NEEDLE` → Missing structural coverage in needle taxonomy.

This cell evaluates **pipeline integrity**, not argument strength.


In [3]:
import pandas as pd
from pathlib import Path

# -------------------------
# INPUTS
# -------------------------
WS_ENHANCED = Path("/home/hello/Projects/Statements/output/Leonardo_WS_enhanced.csv")

# df_flat is the output from flatten_moltie_jsonl(JSONL_PATH)
# It must exist already in the notebook.
assert "df_flat" in globals(), "df_flat not found. Run flatten_moltie_jsonl first."

# -------------------------
# LOAD WS enhanced
# -------------------------
ws = pd.read_csv(WS_ENHANCED)
print("WS enhanced shape:", ws.shape)

# -------------------------
# Identify row id + derive y_row_id if needed
# -------------------------
# Prefer an existing y_row_id if present, else derive from row index.
if "y_row_id" in ws.columns:
    ws["y_row_id"] = ws["y_row_id"].astype(str)
else:
    # Try common row id columns
    if "row_id" in ws.columns:
        row_num = pd.to_numeric(ws["row_id"], errors="coerce")
    elif "X1" in ws.columns:
        row_num = pd.to_numeric(ws["X1"], errors="coerce")
    else:
        # fallback: CSV row order (1-based)
        row_num = pd.Series(range(1, len(ws) + 1), index=ws.index)

    ws["y_row_id"] = row_num.apply(lambda n: f"X1_{int(n):04d}" if pd.notna(n) else None)

# -------------------------
# Detect "has any needle" in WS enhanced
# -------------------------
if "has__any_needle" in ws.columns:
    ws["ws_has_any_needle"] = ws["has__any_needle"].astype(bool)
elif "needle_selected_raw" in ws.columns:
    ws["ws_has_any_needle"] = ws["needle_selected_raw"].notna() & (ws["needle_selected_raw"].astype(str).str.strip() != "")
else:
    # last resort: infer from any has__* column if present
    has_cols = [c for c in ws.columns if c.startswith("has__")]
    if has_cols:
        ws["ws_has_any_needle"] = ws[has_cols].fillna(False).astype(bool).any(axis=1)
    else:
        raise ValueError("Cannot determine needle hits: no has__any_needle, needle_selected_raw, or has__* columns found.")

# -------------------------
# Compute "has any relevant Moltie match" per y_row_id
# -------------------------
m = df_flat.copy()

# Defensive: relevant column may be object
m["relevant"] = m["relevant"] == True

rel_by_row = (
    m.groupby("y_row_id", dropna=False)["relevant"]
    .any()
    .rename("has_relevant_case")
    .reset_index()
)

# -------------------------
# Join + bucket
# -------------------------
ws_row = ws[["y_row_id", "ws_has_any_needle"]].drop_duplicates("y_row_id").copy()
ws_row = ws_row.merge(rel_by_row, on="y_row_id", how="left")
ws_row["has_relevant_case"] = ws_row["has_relevant_case"].fillna(False)

def bucket(r):
    if not r["ws_has_any_needle"]:
        return "NO_NEEDLE"
    if r["has_relevant_case"]:
        return "NEEDLE_AND_RELEVANT"
    return "NEEDLE_BUT_NO_RELEVANT"

ws_row["bucket"] = ws_row.apply(bucket, axis=1)

# -------------------------
# Frequency table
# -------------------------
freq = (
    ws_row["bucket"]
    .value_counts(dropna=False)
    .rename_axis("bucket")
    .reset_index(name="count")
)

freq["pct"] = (freq["count"] / freq["count"].sum() * 100).round(2)

# stable ordering
order = ["NEEDLE_AND_RELEVANT", "NEEDLE_BUT_NO_RELEVANT", "NO_NEEDLE"]
freq["bucket"] = pd.Categorical(freq["bucket"], categories=order, ordered=True)
freq = freq.sort_values("bucket").reset_index(drop=True)

print(freq)

# Optional: show which rows fall in each bucket (for QA)
display(ws_row.sort_values(["bucket", "y_row_id"]).head(50))

WS enhanced shape: (12, 37)
                   bucket  count    pct
0     NEEDLE_AND_RELEVANT      3  25.00
1  NEEDLE_BUT_NO_RELEVANT      1   8.33
2               NO_NEEDLE      8  66.67


/tmp/ipykernel_2269070/2907857589.py:72: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ws_row["has_relevant_case"] = ws_row["has_relevant_case"].fillna(False)


,y_row_id,ws_has_any_needle,has_relevant_case,bucket
9,X1_0010,True,True,NEEDLE_AND_RELEVANT
10,X1_0011,True,True,NEEDLE_AND_RELEVANT
11,X1_0012,True,True,NEEDLE_AND_RELEVANT
7,X1_0008,True,False,NEEDLE_BUT_NO_RELEVANT
0,X1_0001,False,False,NO_NEEDLE
1,X1_0002,False,False,NO_NEEDLE
2,X1_0003,False,False,NO_NEEDLE
3,X1_0004,False,False,NO_NEEDLE
4,X1_0005,False,False,NO_NEEDLE
5,X1_0006,False,False,NO_NEEDLE


# MOLTIE – Prework Cell (Y Summary + JSONL Flatten)

## Purpose

This notebook cell performs **two prep steps** before deeper Moltie analysis:

1. **Summarise `Y_inferred.json`**: which WS rows produced a valid `Y` and which `X` tests were emitted.
2. **Flatten `MASTER_moltie.jsonl`** into tabular form and build a **PDF hit frequency** view.

It then saves both flattened outputs as CSVs next to the master JSONL file.

---

## Inputs

### Files

* `Y_inferred.json`

  * Path: `/home/hello/Projects/Statements/output/Y_inferred.json`
  * Expected to contain a `rows` object keyed by `y_row_id`.

* `MASTER_moltie.jsonl`

  * Path: `/home/hello/Projects/Statements/output/moltie_batch/MASTER_moltie.jsonl`

* `flatter.py`

  * Dir: `/home/hello/Projects/Statements/code/data_analysis`
  * Must export:

    * `flatten_moltie_jsonl(jsonl_path)`
    * `pdf_hit_frequency_view(df_flat)`

### Python deps

* `pandas`

---

## What it Produces

### 1) `y_summary_df`

A per-WS-row summary table:

* `y_row_id`: key from `Y_inferred.json: rows`
* `row_index_1based`: 1-based WS row index
* `y_ok`: whether inference succeeded for that row
* `n_x_tests`: number of `X*` tests emitted (only if `y_ok=True`)
* `x_keys`: list of X test keys (e.g. `['X5', 'X12']`)

Also prints:

* total Y rows
* count of `y_ok` vs `y_fail`

### 2) `df_flat`

Flattened table output from `MASTER_moltie.jsonl` (schema defined by `flatter.py`).

### 3) `df_pdf_freq`

Aggregated PDF-level frequency view computed from `df_flat`.

---

## Output Files

Saved next to `MASTER_moltie.jsonl`:

* `MASTER_moltie__flat.csv`
* `MASTER_moltie__pdf_frequency.csv`

(Implemented by stripping `.jsonl` then appending `__flat.csv` and `__pdf_frequency.csv`.)

---

## Execution Flow

1. **Validate input paths** with `assert`.
2. **Load Y JSON** and build `y_summary_df`:

   * Only extracts `x_tests` when `y_ok=True`.
3. **Import flatteners** by adding `FLATTER_DIR` to `sys.path`.
4. **Generate outputs**:

   * `df_flat = flatten_moltie_jsonl(JSONL_PATH)`
   * `df_pdf_freq = pdf_hit_frequency_view(df_flat)`
5. **Save CSVs** and display:

   * `df_flat.head(10)`
   * `df_pdf_freq.head(20)`

---

## Notes / Assumptions

* `Y_inferred.json` must have structure: `{'rows': {<y_row_id>: {...}}}`.
* `x_tests` is expected to be a dict; if not, it is ignored.
* `display(...)` assumes a Jupyter environment.

---

# MOLTIE – WS ↔ Moltie Relevance Alignment Cell

## Purpose

This cell aligns:

* **Witness Statement enhanced CSV** (`Leonardo_WS_enhanced.csv`)
* **Moltie flattened output** (`df_flat` from `flatten_moltie_jsonl`)

It classifies each WS row into buckets based on:

1. Whether the WS row hit any needle.
2. Whether Moltie found at least one **relevant** case for that row.

---

## Preconditions

* `df_flat` must already exist in memory.
* `df_flat` must contain at least:

  * `y_row_id`
  * `relevant`

---

## Step 1 – Load WS Enhanced CSV

Input:
`/home/hello/Projects/Statements/output/Leonardo_WS_enhanced.csv`

Prints shape for validation.

---

## Step 2 – Ensure `y_row_id`

Priority order:

1. Use existing `y_row_id` column.
2. Else derive from:

   * `row_id`
   * or `X1`
   * or fallback to 1-based row order.

Derived format:

```
X1_0001
X1_0002
...
```

---

## Step 3 – Detect WS Needle Hits

Creates `ws_has_any_needle` using:

Priority:

1. `has__any_needle`
2. `needle_selected_raw`
3. Any column starting with `has__`

If none found → raises error.

---

## Step 4 – Compute Moltie Relevance by Row

From `df_flat`:

* Force `relevant` to boolean.
* Group by `y_row_id`.
* Compute:

```
has_relevant_case = any(relevant == True)
```

Result: one row per `y_row_id`.

---

## Step 5 – Join + Bucket Classification

Merge WS-level view with Moltie relevance.

Each row classified into:

* `NEEDLE_AND_RELEVANT`
* `NEEDLE_BUT_NO_RELEVANT`
* `NO_NEEDLE`

Logic:

* No WS needle → `NO_NEEDLE`
* Needle + relevant case → `NEEDLE_AND_RELEVANT`
* Needle + no relevant case → `NEEDLE_BUT_NO_RELEVANT`

---

## Step 6 – Frequency Table

Produces:

| bucket | count | pct |

With stable ordering:

1. `NEEDLE_AND_RELEVANT`
2. `NEEDLE_BUT_NO_RELEVANT`
3. `NO_NEEDLE`

Also displays first 50 classified rows for QA.

---

## Interpretation

This table answers:

* Are WS needles actually finding precedent support?
* Where do we have needle hits but no relevant Moltie match?
* Where are we completely blind (no needle)?

This is the integrity bridge between:

**WS Enhanced → Needle Logic → Moltie → Precedent Relevance**

---

# MOLTIE – X Metrics + Mostly‑Noise Triage (Single Cell)

## Purpose

Build a single aggregated table (`x_metrics`) that measures how well each **(WS row, X-test)** performs across all ET documents, then derive a second table (`bad`) that flags **mostly-noise** combinations (high run volume, low relevance rate).

## Inputs

* `df_flat` (from `flatten_moltie_jsonl`) with at least:

  * `y_row_id`, `x_key`, `x_name`, `et_path`, `relevant`, `confidence`

## Outputs

* `x_metrics`: one row per (`y_row_id`, `x_key`, `x_name`) with:

  * `total_runs`, `n_relevant`, `relevant_rate_pct`, plus max signal fields.
  * Sorted to show strongest performers first.
* `bad`: subset of `x_metrics` where:

  * `total_runs >= MIN_RUNS` and `relevant_rate_pct <= MAX_RATE_PCT`
  * Includes `bad_score` to prioritise the most expensive low‑yield pairs.

## What it prints/displays

* Top 50 rows of `x_metrics`
* Counts: universe size + mostly‑noise row count
* Top 50 rows of `bad`


In [4]:
import numpy as np
import pandas as pd

# =========================================================
# X METRICS (GOOD SURFACE) + MOSTLY-NOISE TRIAGE (BAD SURFACE)
# =========================================================

assert "df_flat" in globals(), "df_flat not found. Run flatten_moltie_jsonl first."

# -------------------------
# 1) Build x_metrics (one row per WS row + X test)
# -------------------------
m = df_flat.copy()
m["relevant"] = m["relevant"] == True  # normalize

x_metrics = (
    m.groupby(["y_row_id", "x_key", "x_name"], dropna=False)
     .agg(
         total_runs=("et_path", "size"),
         n_relevant=("relevant", "sum"),
         max_confidence=("confidence", "max"),
         max_precedent_score=("precedent_score", "max"),
         max_anchor_count=("anchor_count", "max"),
     )
     .reset_index()
)

x_metrics["relevant_rate_pct"] = (
    x_metrics["n_relevant"] / x_metrics["total_runs"] * 100
).round(2)

# Sort: strongest first
x_metrics = x_metrics.sort_values(
    by=["relevant_rate_pct", "n_relevant", "max_precedent_score"],
    ascending=[False, False, False],
    na_position="last"
).reset_index(drop=True)

display(x_metrics.head(50))

# -------------------------
# 2) Mostly-noise triage from x_metrics
# -------------------------
MIN_RUNS = 10
MAX_RATE_PCT = 10.0   # <=10% relevant is mostly-noise

bad = x_metrics[
    (x_metrics["total_runs"] >= MIN_RUNS) &
    (x_metrics["relevant_rate_pct"] <= MAX_RATE_PCT)
].copy()

# Score: wasted calls * (confidence boost if it keeps firing "confidently")
bad["max_confidence"] = pd.to_numeric(bad["max_confidence"], errors="coerce").fillna(0)
conf01 = (bad["max_confidence"] / 100.0).clip(0, 1)

bad["bad_score"] = (
    np.log1p(bad["total_runs"]) *
    (1 - bad["relevant_rate_pct"] / 100.0) *
    (1 + 0.6 * conf01)
)

bad = bad.sort_values(
    by=["bad_score", "total_runs", "relevant_rate_pct"],
    ascending=[False, False, True],
    na_position="last"
).reset_index(drop=True)

print("Universe rows:", len(x_metrics))
print("Mostly-noise bad rows:", len(bad))
display(bad.head(50))

,y_row_id,x_key,x_name,total_runs,n_relevant,max_confidence,max_precedent_score,max_anchor_count,relevant_rate_pct
0,X1_0010,X6,Contextual Evidence of Procedural Failures,321,279,80.0,80.0,3,86.92
1,X1_0010,X7,Lack of Fair and Open-Minded Process,321,255,80.0,80.0,3,79.44
2,X1_0010,X2,Failure to Address Internal Dissemination,321,169,80.0,80.0,3,52.65
3,X1_0011,X2,Proportionality of Dismissal,2344,1145,80.0,80.0,3,48.85
4,X1_0010,X3,Predetermination Evidence,321,151,80.0,80.0,3,47.04
5,X1_0012,X4,Inconsistent and Unreasonable Reasoning,2577,1066,80.0,80.0,3,41.37
6,X1_0012,X1,Failure to Provide Timely Resolution,2577,1057,80.0,80.0,4,41.02
7,X1_0012,X2,Lack of Genuine Reconsideration,2577,971,80.0,80.0,3,37.68
8,X1_0010,X4,Breach of Confidentiality,322,77,80.0,80.0,3,23.91
9,X1_0010,X1,Premature Disclosure of Allegations,321,75,80.0,80.0,3,23.36


Universe rows: 17
Mostly-noise bad rows: 1


,y_row_id,x_key,x_name,total_runs,n_relevant,max_confidence,max_precedent_score,max_anchor_count,relevant_rate_pct,bad_score
0,X1_0011,X5,Comparator Evidence,2344,118,80.0,80.0,4,5.03,10.907172
